# 1. this here is for preprcessing the data and clean it

In [17]:
import pandas as pd
import numpy
from pathlib import Path

In [18]:
p = Path('../train-template-kit') / 'train.csv'
df = pd.read_csv(p)
print(df.head())

   booking_id          hotel  cancelled  days_ahead  arrival_year  \
0           2  Seaside Hotel          0           7          2015   
1           3  Seaside Hotel          0          13          2015   
2           4  Seaside Hotel          0          14          2015   
3           5  Seaside Hotel          0          14          2015   
4           6  Seaside Hotel          0           0          2015   

  arrival_month  arrival_week  arrival_day  weekend_nights  weekday_nights  \
0          July            27            1               0               1   
1          July            27            1               0               1   
2          July            27            1               0               2   
3          July            27            1               0               2   
4          July            27            1               0               2   

   ...     deposit  agent  company waitlist_days guest_type nightly_rate  \
0  ...  No Deposit    NaN      NaN      

## 2. Country breakdown

Key findings (train.csv, arrivals 2015-07 to 2016-12):

- **160 country codes, but very concentrated.** The top 5 (PRT, GBR, FRA, ESP, DEU) cover 75% of rows and the top 20 cover 94%. 73 codes have fewer than 10 rows.
- **Portugal dominates the cancellations.** It is 45% of all requests and 57% of them are cancelled, against 13–22% for GBR, FRA, ESP and DEU. PRT also holds almost all the `Non Refund` deposits (29% of PRT rows). Excluding those, PRT is still 40% cancelled.
- **PRT is worth the least per request.** Realised revenue per request is about €113 for PRT, against €290–380 for GBR, FRA, IRL, NLD and CHE. PRT makes up 45% of requests but only 25% of the revenue from guests who showed up.
- **The Portugal effect is partly an artefact.** Among cancelled bookings, the share of Portugal falls from 93% (2015 Q3) to 54% (2016 Q4), while for guests who showed up it falls from 41% to 23%. In the source data, nationality is often only confirmed at check-in, so cancelled bookings tend to default to Portugal. It still works as a feature, but the effect is weaker in recent data, so expect it to be weaker again in 2017.
- **Other countries with high cancellation:** AGO 57%, CHN 45%, TUR 40%, MAR 36%, ITA 32%, BRA 32%. **Low:** DEU, AUT, FIN about 13%; FRA 15%.
- **`CN` is a different country from `CHN`.** It has 640 rows, mostly at Seaside, 4-night stays and 16% cancelled; `CHN` (China) has 564 rows, mostly Downtown and 45% cancelled. `CAN` never appears, so `CN` is probably Canada. Keep the two codes separate.
- **Missing country (395 rows):** almost all Seaside, only 13% cancelled, median lead time 5 days, median rate €48. Treat it as its own category.
- **Seaside cancels less for every country** (e.g. GBR 9% vs 27% at Downtown), so model the hotel together with the country.

In [19]:
import numpy as np

df = pd.read_csv("../train-template-kit/train.csv", parse_dates=["arrival_date", "booking_date"])
df["country"] = df["country"].fillna("(missing)")
df["stay_value"] = df["nightly_rate"] * df["nights"]
df["show_value"] = df["stay_value"] * (1 - df["cancelled"])  # revenue actually earned
df.shape

(78132, 34)

In [20]:
# How concentrated are the countries?
vc = df["country"].value_counts()
print(f"{vc.size} country codes")
for k in (5, 10, 20):
    print(f"top {k:>2}: {vc.head(k).sum() / len(df):.1%} of rows")
print(f"{(vc < 10).sum()} codes with <10 rows ({vc[vc < 10].sum()} rows), "
      f"{(vc < 100).sum()} codes with <100 rows ({vc[vc < 100].sum()} rows)")

160 country codes
top  5: 75.4% of rows
top 10: 86.0% of rows
top 20: 94.3% of rows
73 codes with <10 rows (201 rows), 127 codes with <100 rows (1951 rows)


### Top 20 countries + the rest

`revenue_per_request` = average revenue from guests who showed up (rate × nights, 0 if cancelled), per request. `revenue_share` = this country's share of all that revenue.

In [21]:
TOP_N = 20
top = vc.head(TOP_N).index
df["country_grp"] = df["country"].where(df["country"].isin(top), "(other)")
non_refund = df["deposit"].eq("Non Refund")

by_country = (
    df.assign(non_refund=non_refund, downtown=df["hotel"].eq("Downtown Hotel"))
    .groupby("country_grp")
    .agg(
        rows=("cancelled", "size"),
        cancel_rate=("cancelled", "mean"),
        non_refund_share=("non_refund", "mean"),
        downtown_share=("downtown", "mean"),
        median_rate=("nightly_rate", "median"),
        median_nights=("nights", "median"),
        median_lead=("days_ahead", "median"),
        revenue_per_request=("show_value", "mean"),
        show_value=("show_value", "sum"),
    )
)
by_country.insert(1, "row_share", by_country["rows"] / len(df))
by_country.insert(3, "cancel_rate_excl_non_refund", df[~non_refund].groupby("country_grp")["cancelled"].mean())
by_country["revenue_share"] = by_country.pop("show_value") / df["show_value"].sum()
by_country.sort_values("rows", ascending=False).round(3)

,rows,row_share,cancel_rate,cancel_rate_excl_non_refund,non_refund_share,downtown_share,median_rate,median_nights,median_lead,revenue_per_request,revenue_share
country_grp,,,,,,,,,,,
PRT,35150,0.450,0.568,0.396,0.286,0.648,80.00,2.0,75.0,112.865,0.245
GBR,6917,0.089,0.162,0.146,0.020,0.383,80.75,4.0,94.0,345.390,0.148
FRA,6502,0.083,0.145,0.144,0.002,0.840,95.00,3.0,54.0,287.301,0.115
ESP,5941,0.076,0.220,0.203,0.021,0.547,96.00,3.0,33.0,257.097,0.094
(other),4460,0.057,0.264,0.265,0.001,0.795,99.45,3.0,41.0,265.873,0.073
DEU,4388,0.056,0.133,0.135,0.010,0.816,95.40,3.0,101.0,269.648,0.073
ITA,2448,0.031,0.321,0.321,0.000,0.899,100.00,3.0,69.0,227.245,0.034
IRL,1971,0.025,0.194,0.194,0.000,0.292,80.75,5.0,96.0,379.942,0.046
BEL,1341,0.017,0.167,0.158,0.011,0.800,99.80,3.0,62.0,308.188,0.026


In [22]:
# Cancellation rate per hotel for the top 10 countries (Seaside is lower everywhere)
top10 = vc.head(10).index
(
    df[df["country"].isin(top10)]
    .pivot_table(index="country", columns="hotel", values="cancelled", aggfunc=["mean", "size"])
    .round(3)
)

mean                         size              
hotel   Downtown Hotel Seaside Hotel Downtown Hotel Seaside Hotel
country                                                          
BEL              0.180         0.116           1073           268
BRA              0.347         0.197            981           229
DEU              0.146         0.077           3580           808
ESP              0.276         0.152           3249          2692
FRA              0.159         0.071           5461          1041
GBR              0.272         0.093           2647          4270
IRL              0.332         0.138            576          1395
ITA              0.343         0.121           2201           247
NLD              0.189         0.060            951           349
PRT              0.638         0.441          22785         12365

In [23]:
# Extremes among countries with at least 50 rows
rates = df.groupby("country")["cancelled"].agg(cancel_rate="mean", rows="size")
rates = rates[rates["rows"] >= 50].sort_values("cancel_rate")
display(rates.head(8).round(3), rates.tail(8).round(3))

,cancel_rate,rows
country,,
SRB,0.012,85
LTU,0.054,56
(missing),0.129,395
AUT,0.133,775
DEU,0.133,4388
FIN,0.135,281
JPN,0.139,122
FRA,0.145,6502


,cancel_rate,rows
country,,
KOR,0.319,69
ITA,0.321,2448
HUN,0.326,129
MAR,0.358,151
TUR,0.395,177
CHN,0.445,564
AGO,0.565,237
PRT,0.568,35150


### Is the Portugal effect real or an artefact?

Compare PRT with the rest *within* the same segment and lead-time band (`Non Refund` bookings excluded). If the gap were only caused by the mix of segments and lead times, it would disappear here. It does not, but it shrinks over time (last table).

In [24]:
no_dep = df[~non_refund].assign(is_prt=lambda d: d["country"].eq("PRT"))
display(
    no_dep.pivot_table(index="sales_segment", columns="is_prt", values="cancelled", aggfunc=["mean", "size"]).round(3),
    no_dep.pivot_table(index=pd.cut(no_dep["days_ahead"], [-1, 7, 30, 90, 180, 800]), columns="is_prt",
                       values="cancelled", aggfunc="mean", observed=True).round(3),
)

mean            size        
is_prt         False  True     False   True 
sales_segment                               
Aviation       0.071  0.436     70.0    55.0
Complementary  0.021  0.120     47.0   465.0
Corporate      0.033  0.165    879.0  2662.0
Direct         0.094  0.210   3874.0  3993.0
Groups         0.026  0.584   3672.0  3866.0
Offline TA/TO  0.017  0.432   9036.0  5527.0
Online TA      0.300  0.462  25005.0  8518.0
Undefined        NaN  1.000      NaN     2.0

is_prt,False,True
days_ahead,,
"(-1, 7]",0.059,0.132
"(7, 30]",0.197,0.345
"(30, 90]",0.233,0.466
"(90, 180]",0.245,0.521
"(180, 800]",0.160,0.716


In [25]:
# Share of PRT among cancelled vs shown bookings, by arrival quarter
(
    df.assign(quarter=df["arrival_date"].dt.to_period("Q"), is_prt=df["country"].eq("PRT"))
    .pivot_table(index="quarter", columns="cancelled", values="is_prt", aggfunc="mean")
    .rename(columns={0: "prt_share_shown", 1: "prt_share_cancelled"})
    .round(3)
)

cancelled,prt_share_shown,prt_share_cancelled
quarter,,
2015Q3,0.410,0.926
2015Q4,0.406,0.939
2016Q1,0.380,0.718
2016Q2,0.267,0.704
2016Q3,0.216,0.511
2016Q4,0.225,0.538


In [26]:
# CN vs CHN: two different populations (CAN never appears -> CN is probably Canada)
(
    df[df["country"].isin(["CN", "CHN", "(missing)"])]
    .groupby("country")
    .agg(rows=("cancelled", "size"), cancel_rate=("cancelled", "mean"),
         downtown_share=("hotel", lambda h: h.eq("Downtown Hotel").mean()),
         median_nights=("nights", "median"), median_lead=("days_ahead", "median"),
         median_rate=("nightly_rate", "median"))
    .round(3)
)

,rows,cancel_rate,downtown_share,median_nights,median_lead,median_rate
country,,,,,,
(missing),395,0.129,0.043,2.0,5.0,48.0
CHN,564,0.445,0.874,2.0,28.0,107.1
CN,640,0.164,0.453,4.0,81.0,91.0


### What this means for preprocessing

- Encode `country` as a category: keep the top ~20 codes, group the rest into `(other)`, and keep `(missing)` separate (it behaves differently).
- Add an `is_prt` flag. It is the strongest country signal, but it gets weaker over time, so weight 2016 more.
- Keep `CN` and `CHN` separate.
- Combine with `hotel`, since the cancellation rate per country differs strongly between the hotels.

## 3. Cleaning pipeline

The cleaning code lives in **`prep.py`**, not in this notebook, because `Solution.decide()` must run exactly the same code on every 2017 booking. This notebook runs it on `train.csv`, shows what changed, checks it, and saves the result.

- Rules and reasons: `data_preparation.md`
- Short overview of the steps: `cleaning_steps.md`
- Outputs:
  - `data/train_clean.parquet`: the cleaned training table
  - `data/prep_params.json`: the fitted parameters the Solution loads

In [27]:
%load_ext autoreload
%autoreload 2
import time
from pathlib import Path

import prep

RAW_PATH = Path("../train-template-kit/train.csv")
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

raw = prep.load_raw(RAW_PATH)
raw.shape

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


(78132, 32)

### 3.1 Fit the cleaning parameters

These are learned from training data only:

- Which categories to keep: top 20 countries, and agents, companies and room types with at least 50 rows. Rarer values become `other`.
- A rate cap per hotel (99.9th percentile), used for features only.
- A reference rate per hotel and arrival month, taken from the latest year (for `rate_rel`).

These parameters are fitted on all data and are what the Solution uses. Section 4 fits a separate set on train + valid only, for the splits.

In [28]:
params = prep.fit_params(raw)
print("rate cap per hotel:", params["rate_cap"])
print("categories kept per column:", {col: len(v) for col, v in params["categories"].items()})
print("countries kept:", params["categories"]["country"])

rate cap per hotel: {'Downtown Hotel': 266.81, 'Seaside Hotel': 327.49}
categories kept per column: {'hotel': 3, 'meal': 5, 'country': 22, 'sales_segment': 8, 'sales_channel': 5, 'room_type': 9, 'deposit': 4, 'guest_type': 5, 'agent': 101, 'company': 18}
countries kept: ['(missing)', '(other)', 'AUT', 'BEL', 'BRA', 'CHE', 'CHN', 'CN', 'DEU', 'ESP', 'FRA', 'GBR', 'IRL', 'ISR', 'ITA', 'NLD', 'NOR', 'POL', 'PRT', 'RUS', 'SWE', 'USA']


### 3.2 Clean the values and add features

`prepare()` does three things:

1. Counts identical requests already seen on the same booking date (`same_request_count`).
2. Fixes values and maps categories.
3. Adds the derived features.

`add_training_columns()` then adds three columns used for training only: `drop_for_training`, `use_for_occupancy` and `split`.

In [29]:
df = prep.add_training_columns(prep.prepare(raw, params))
df[prep.FEATURES].head()

,days_ahead,arrival_week,arrival_day,weekend_nights,weekday_nights,nights,adults,children,babies,total_guests,...,hotel,meal,country,sales_segment,sales_channel,room_type,deposit,guest_type,agent,company
0,7,27,1,0,1,1,1,0.0,0,1.0,...,Seaside Hotel,BB,GBR,Direct,Direct,A,No Deposit,Transient,none,none
1,13,27,1,0,1,1,1,0.0,0,1.0,...,Seaside Hotel,BB,GBR,Corporate,Corporate,A,No Deposit,Transient,other,none
2,14,27,1,0,2,2,2,0.0,0,2.0,...,Seaside Hotel,BB,GBR,Online TA,TA/TO,A,No Deposit,Transient,240,none
3,14,27,1,0,2,2,2,0.0,0,2.0,...,Seaside Hotel,BB,GBR,Online TA,TA/TO,A,No Deposit,Transient,240,none
4,0,27,1,0,2,2,2,0.0,0,2.0,...,Seaside Hotel,BB,PRT,Direct,Direct,C,No Deposit,Transient,none,none


### 3.3 What changed

In [30]:
changes = pd.Series({
    "children missing -> 0": raw["children"].isna().sum(),
    "children > 3 -> 3": (raw["children"] > 3).sum(),
    "babies > 2 -> 2": (raw["babies"] > 2).sum(),
    "meal Undefined -> SC": raw["meal"].eq("Undefined").sum(),
    "segment / channel Undefined -> Other": (raw["sales_segment"].eq("Undefined")
                                             | raw["sales_channel"].eq("Undefined")).sum(),
    "country missing -> (missing)": raw["country"].isna().sum(),
    "country rare -> (other)": df["country"].eq("(other)").sum(),
    "agent blank -> none": raw["agent"].isna().sum(),
    "agent rare -> other": df["agent"].eq("other").sum(),
    "company blank -> none": raw["company"].isna().sum(),
    "company rare -> other": df["company"].eq("other").sum(),
    "room_type rare -> other": df["room_type"].eq("other").sum(),
    "rate capped (feature only)": (df["rate_capped"] < df["nightly_rate"]).sum(),
}, name="rows affected")
changes.to_frame()

,rows affected
children missing -> 0,4
children > 3 -> 3,1
babies > 2 -> 2,2
meal Undefined -> SC,754
segment / channel Undefined -> Other,5
country missing -> (missing),395
country rare -> (other),4153
agent blank -> none,10513
agent rare -> other,2252
company blank -> none,73715


### 3.4 Removed rows and training flags

- **Removed (training table only):** the 13 placeholder rows with ≥ 10 adults and the €5,400 booking.
- **`use_for_occupancy = False`:** arrivals before 2015-07-15, and Downtown arrivals before 2015-09-01 (that period looks incomplete). Leave these out of occupancy and demand estimates only.
- **`split`:** arrivals before 2016-07-01 are `train`, before 2016-10-01 `valid`, later ones `test` (see section 4).

In [31]:
display(df.loc[df["drop_for_training"], ["booking_id", "hotel", "arrival_date", "adults", "nights",
                                         "nightly_rate", "cancelled", "sales_segment"]])
print("use_for_occupancy = False:", (~df["use_for_occupancy"]).sum(), "rows")
df[~df["drop_for_training"]].groupby("split").agg(
    rows=("cancelled", "size"), cancel_rate=("cancelled", "mean"),
    first_arrival=("arrival_date", "min"), last_arrival=("arrival_date", "max"),
).round(3)

,booking_id,hotel,arrival_date,adults,nights,nightly_rate,cancelled,sales_segment
1509,1539,Seaside Hotel,2015-09-03,40,3,0.0,1,Direct
1557,1587,Seaside Hotel,2015-09-05,26,7,0.0,1,Offline TA/TO
1612,1643,Seaside Hotel,2015-09-07,50,3,0.0,1,Direct
1721,1752,Seaside Hotel,2015-09-12,26,7,0.0,1,Offline TA/TO
1852,1884,Seaside Hotel,2015-09-19,26,7,0.0,1,Offline TA/TO
1885,1917,Seaside Hotel,2015-09-21,27,4,0.0,1,Direct
1930,1962,Seaside Hotel,2015-09-24,27,4,0.0,1,Direct
1971,2003,Seaside Hotel,2015-09-26,26,7,0.0,1,Offline TA/TO
2132,2164,Seaside Hotel,2015-10-03,26,7,0.0,1,Offline TA/TO
2141,2173,Seaside Hotel,2015-10-04,55,2,0.0,1,Direct


use_for_occupancy = False: 4438 rows


/tmp/ipykernel_31073/2496295068.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,rows,cancel_rate,first_arrival,last_arrival
split,,,,
test,14378,0.386,2016-10-01,2016-12-31
train,48772,0.360,2015-07-01,2016-06-30
valid,14968,0.357,2016-07-01,2016-09-30


### 3.5 Checks

In [32]:
# Basic checks on the whole table
X = df[prep.FEATURES]
assert len(df) == len(raw)
assert X.notna().all().all(), X.isna().sum()[X.isna().sum() > 0]
assert df["nightly_rate"].equals(raw["nightly_rate"]), "raw rate must stay untouched (revenue)"
assert not set(prep.NOT_FEATURES) & set(prep.FEATURES)
print(f"{len(prep.FEATURES)} features ({len(prep.NUMERIC_FEATURES)} numeric, "
      f"{len(prep.CATEGORICAL_FEATURES)} categorical): all checks passed")

41 features (31 numeric, 10 categorical): all checks passed


In [33]:
# A 2017 booking with values never seen in training must still be handled cleanly
fake = raw.drop(columns="cancelled").iloc[0].to_dict()
fake.update(country="XYZ", agent=99999.0, company=None, meal="Undefined", children=float("nan"), room_type="Z")
out = prep.prepare(fake, params, counter=prep.RequestCounter()).iloc[0]
assert (out["country"], out["agent"], out["company"], out["meal"], out["children"], out["room_type"]) == \
       ("(other)", "other", "none", "SC", 0, "other")
out[["country", "agent", "company", "meal", "children", "room_type"]]

country      (other)
agent          other
company         none
meal              SC
children         0.0
room_type      other
Name: 0, dtype: object

In [34]:
# decide() sees one booking at a time (dict, Timestamps, no `cancelled`).
# Online mode must give exactly the same features as batch mode.
stream = raw.sort_values(["booking_date", "booking_id"], kind="stable").iloc[:1500]
counter = prep.RequestCounter()
start = time.perf_counter()
online = pd.concat([prep.prepare(row, params, counter=counter)
                    for row in stream.drop(columns="cancelled").to_dict("records")])
per_row_ms = (time.perf_counter() - start) / len(stream) * 1000
batch = prep.prepare(stream, params)
pd.testing.assert_frame_equal(online[prep.FEATURES].reset_index(drop=True),
                              batch.loc[stream.index, prep.FEATURES].reset_index(drop=True), check_dtype=False)
print(f"online == batch for {len(stream)} bookings; {per_row_ms:.1f} ms per booking "
      f"(~{per_row_ms * 40_000 / 60_000:.0f} min for 40k decide() calls)")

online == batch for 1500 bookings; 27.9 ms per booking (~19 min for 40k decide() calls)


In [35]:
# The new same_request_count feature: big identical blocks cancel a lot
df.groupby(pd.cut(df["same_request_count"], [-1, 0, 1, 4, 9, 49, 1000]), observed=True)["cancelled"] \
  .agg(cancel_rate="mean", rows="size").round(3)

,cancel_rate,rows
same_request_count,,
"(-1, 0]",0.253,53872
"(0, 1]",0.375,5553
"(1, 4]",0.510,5028
"(4, 9]",0.643,4431
"(9, 49]",0.777,8300
"(49, 1000]",0.908,948


### 3.6 Save

In [36]:
train_clean = df[~df["drop_for_training"]].reset_index(drop=True)
train_clean.to_parquet(OUT_DIR / "train_clean.parquet", index=False)
prep.save_params(params, OUT_DIR / "prep_params.json")

check = pd.read_parquet(OUT_DIR / "train_clean.parquet")
assert check.shape == train_clean.shape and check["country"].dtype == "category"
print("saved", train_clean.shape, "->", OUT_DIR / "train_clean.parquet", "and", OUT_DIR / "prep_params.json")

saved (78118, 53) -> data/train_clean.parquet and data/prep_params.json


### 3.7 How the model and the Solution use this

```python
# training
train = pd.read_parquet("data/train_clean.parquet")
X, y = train[prep.FEATURES], train[prep.TARGET]          # y = 1 means cancelled

# Solution.__init__
self.params = prep.load_params("data/prep_params.json")
self.counter = prep.RequestCounter()

# Solution.decide(booking)
x = prep.prepare(booking, self.params, counter=self.counter)[prep.FEATURES]
```

- Every feature is the same whatever we decide, so a local replay can prepare all requests in one go, which is much faster. Only the real `evaluate()` needs the slower one-booking-at-a-time mode.
- `prep.HINDSIGHT_FEATURES` lists the columns recorded after booking time. Train once with and once without them to see how much they matter.

## 4. Train / validate / test split

We split by **arrival date**, never randomly. A random split would put identical group bookings on both sides, and the data changes over time.

| Split | Arrivals | Use |
|---|---|---|
| `train` | Jul 2015 – Jun 2016 (1 year) | Fit models |
| `valid` | Jul – Sep 2016 (¼ year) | Compare models and choose settings (use as often as you like) |
| `test` | Oct – Dec 2016 (¼ year) | Look at it **rarely**: final check of the model and replay of the accept/reject rule |

- **Output:** `data/splits/train.parquet`, `valid.parquet` and `test.parquet`. **One data source for both hotels:** `prep.load_split(name, hotel="downtown")` loads one hotel only, so both ways work: one model per hotel, or one joint model to compare.
- The cleaning parameters are fitted on **train + valid only** (`until=prep.TEST_START`), so the test period has no influence. They are saved as `data/prep_params_dev.json`.
- `is_commitment` (valid/test only): `True` if the booking was made before its split started. In a replay of that split these are the confirmed bookings; the other rows are the requests that go to `decide()`.
- The 14 broken rows are removed from train (valid and test have none).
- What every added column means: `docs/cleaning_steps.md`.
- **For the submission**, retrain on all data using `data/prep_params.json` (section 3).

In [37]:
dev_params = prep.fit_params(raw, until=prep.TEST_START)
prep.save_params(dev_params, OUT_DIR / "prep_params_dev.json")

full = prep.add_training_columns(prep.prepare(raw, dev_params))
full = full[~(full["drop_for_training"] & full["split"].ne("test"))]  # broken rows only matter for training
full = full.drop(columns="drop_for_training").reset_index(drop=True)    # always False from here on

prep.SPLIT_DIR.mkdir(parents=True, exist_ok=True)
splits = {}
for name in ("train", "valid", "test"):
    splits[name] = full[full["split"] == name].reset_index(drop=True)
    splits[name].to_parquet(prep.SPLIT_DIR / f"{name}.parquet", index=False)
print("written:", sorted(p.name for p in prep.SPLIT_DIR.glob("*.parquet")))

written: ['test.parquet', 'train.parquet', 'valid.parquet']


In [38]:
# Overview of the three splits
summary = pd.DataFrame({
    name: {
        "rows": len(part),
        "share": len(part) / len(full),
        "first_arrival": part["arrival_date"].min().date(),
        "last_arrival": part["arrival_date"].max().date(),
        "cancel_rate": part["cancelled"].mean(),
        "commitments": part["is_commitment"].sum() if name != "train" else None,
        "cancel_rate_commitments": part.loc[part["is_commitment"], "cancelled"].mean() if name != "train" else None,
        "requests": (~part["is_commitment"]).sum() if name != "train" else None,
        "cancel_rate_requests": part.loc[~part["is_commitment"], "cancelled"].mean() if name != "train" else None,
    }
    for name, part in splits.items()
})
summary

,train,valid,test
rows,48772,14968,14378
share,0.624338,0.191608,0.184055
first_arrival,2015-07-01,2016-07-01,2016-10-01
last_arrival,2016-06-30,2016-09-30,2016-12-31
cancel_rate,0.35992,0.357028,0.38552
commitments,None,10359,8921
cancel_rate_commitments,None,0.407665,0.489183
requests,None,4609,5457
cancel_rate_requests,None,0.24322,0.216053


In [39]:
# Checks: nothing lost, no overlap, the files load back
assert sum(len(p) for p in splits.values()) == len(full)
assert splits["train"]["arrival_date"].max() < prep.VALID_START <= splits["valid"]["arrival_date"].min()
assert splits["valid"]["arrival_date"].max() < prep.TEST_START <= splits["test"]["arrival_date"].min()
assert splits["test"]["is_commitment"].any() and not splits["train"]["is_commitment"].any()
for name, part in splits.items():
    back = prep.load_split(name)
    assert back.shape == part.shape and back["country"].dtype == "category"
print("splits OK:", {name: len(part) for name, part in splits.items()})

splits OK: {'train': 48772, 'valid': 14968, 'test': 14378}


### How to use the splits

```python
import prep
train = prep.load_split("train")                         # both hotels (joint model)
train_dt = prep.load_split("train", hotel="downtown")   # one hotel: "downtown" or "seaside"
X_train, y_train = train[prep.FEATURES], train[prep.TARGET]   # y = 1 means cancelled
```

Or read a file directly, e.g. `pd.read_parquet("data/splits/train.parquet")`, and filter on `hotel` yourself.

1. **Develop:** train on `train`, evaluate on `valid`. Check calibration, log-loss and AUC, separately for `is_commitment` True and False.
2. **Test check (rarely):** retrain on `train` + `valid`, then evaluate on `test`. For the replay:
   - Confirmed bookings: `test[test.is_commitment]`.
   - Requests: the remaining rows, sorted by `booking_date`.
   - Capacity: 124 / 103 rooms, minus guests who arrived before 2016-10-01, showed up, and are still staying.
3. **Submission:** retrain on all data (`data/train_clean.parquet`, parameters in `data/prep_params.json`).

### 4.1 One model per hotel

The two hotels behave very differently: Downtown cancels 41%, Seaside 27%, and they have different seasons and guest markets. So we can train **one model per hotel**:

```python
train_dt = prep.load_split("train", hotel="downtown")   # or hotel="seaside"
X, y = train_dt[prep.FEATURES_PER_HOTEL], train_dt[prep.TARGET]   # `hotel` left out (constant)
```

- Same files and same cleaning; `load_split` only filters by hotel.
- `decide()` then picks the model by `booking["hotel"]`.
- Each model has less data (Seaside especially). Compare on `valid` against one joint model that uses `hotel` as a feature (`prep.FEATURES`), and keep whichever wins.

In [40]:
per_hotel = {}
for hotel in prep.HOTELS:
    for name in ("train", "valid", "test"):
        part = prep.load_split(name, hotel=hotel)
        assert part["hotel"].nunique() == 1
        per_hotel[(hotel, name)] = {
            "rows": len(part),
            "cancel_rate": round(part["cancelled"].mean(), 3),
            "commitments": int(part["is_commitment"].sum()) if name != "train" else None,
            "requests": int((~part["is_commitment"]).sum()) if name != "train" else None,
        }

# Both hotels together must give back exactly the full split
for name in ("train", "valid", "test"):
    assert sum(per_hotel[(h, name)]["rows"] for h in prep.HOTELS) == len(splits[name])
pd.DataFrame(per_hotel).T

rows  cancel_rate  commitments  requests
downtown train  31476.0        0.418          NaN       NaN
         valid  10351.0        0.373       7057.0    3294.0
         test    9747.0        0.449       6458.0    3289.0
seaside  train  17296.0        0.254          NaN       NaN
         valid   4617.0        0.322       3302.0    1315.0
         test    4631.0        0.252       2463.0    2168.0